# LoRA vs KronA — fast Unsloth fine-tune

Fine-tune a small chat model **twice** — once with **LoRA**, once with **KronA** — then
compare them apples-to-apples, on **Unsloth**.

> **KRONA = KronA** = the Kronecker adapter (arXiv:2212.10650). In HuggingFace PEFT it is the
> **`LoKr`** adapter (`LoKrConfig`) — that is what we train.

## This notebook is a thin wrapper

All the logic lives in the **`krona/` package** so it is testable and maintainable — no
copy-pasted code between notebook, CLI and future runs:

| | |
|---|---|
| `run.py` | runs the whole experiment from the CLI |
| `compare.py` | diffs the `results.json` of two runs |
| `krona/config.py` | configuration (defaults + `.env` overrides) |
| `krona/env.py` | Colab/RunPod detection, GPU report, HF login |
| `krona/data.py` | dolly-15k → chat-rendered `text` column |
| `krona/adapters.py` | PEFT adapter construction (LoRA and LoKr) |
| `krona/train.py` | model loading + the shared training loop |
| `krona/report.py` | comparison table, samples, `results.json` |
| `scripts/setup_runpod.sh` | one-shot RunPod setup |
| `docs/NOTES.md` | **read this** — the hard-won gotchas |

The notebook just calls the package, so anything you run here you can also run with
`python run.py`.


## Before you run

1. **GPU runtime.** Colab: `Runtime → Change runtime type → T4 GPU`. RunPod: pick a GPU pod.
2. **Start from the repo root**, or set `KRONA_REPO` to the git URL so the next cell clones it.
3. A Hugging Face token is **optional** — `unsloth/gemma-3-1b-it` is public. Put `HF_TOKEN` in
   `.env` (or a Colab secret) if you switch to a gated model.

Then run the cells top to bottom.

> **Known hard constraint:** KronA needs **16-bit**, not 4-bit. See `docs/NOTES.md`.


## 1. Set up the repo & install


In [ ]:
# 1) Locate the repo, install dependencies
import os, sys, pathlib, subprocess

REPO = os.environ.get("KRONA_REPO", "")   # e.g. https://github.com/<you>/Finetuning_LORA_and_KRONA.git

def find_repo():
    """Walk up from the cwd looking for the package; else clone KRONA_REPO."""
    here = pathlib.Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "krona" / "config.py").is_file():
            return candidate
    if not REPO:
        raise SystemExit(
            "Could not find the repo (no krona/config.py above the cwd).\n"
            "Either run this notebook from the repo root, or set KRONA_REPO:\n"
            "    os.environ['KRONA_REPO'] = 'https://github.com/<you>/Finetuning_LORA_and_KRONA.git'"
        )
    dest = pathlib.Path("/content/Finetuning_LORA_and_KRONA") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "Finetuning_LORA_and_KRONA"
    if not dest.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(dest)], check=True)
    return dest

ROOT = find_repo()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))          # so `import krona` works from any cwd
print("repo:", ROOT)

# Unsloth pins its own transformers/peft/trl/bitsandbytes; we only list top-level deps.
print(subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                      str(ROOT / "requirements.txt")], check=True).returncode)


## 2. Configure & check the environment

The configuration comes from `krona/config.py` defaults, overridable from `.env` (prefix
`KRONA_`) or the cell below. On RunPod the default output directory becomes
`/workspace/outputs`, which lives on the network volume and survives a pod restart.


In [ ]:
# 2) Config + environment check
from krona.config import Config
from krona.env import check_dependencies, hf_login, print_environment, require_gpu

cfg = Config.from_env()

# Notebook overrides (same knobs as `python run.py --steps 200`). Comment out to use .env.
cfg.max_samples = 1000
cfg.eval_size   = 64
cfg.steps       = 60
cfg.batch_size  = 2
cfg.grad_accum  = 4
# cfg.model = "unsloth/gemma-3-4b-it"   # bigger GPU? raise the model, not the precision

print(cfg)
print_environment()
require_gpu()
check_dependencies()

# Login is optional: gemma-3-1b-it is public. Set HF_ALLOW_ANONYMOUS to skip the prompt.
os.environ.setdefault("HF_ALLOW_ANONYMOUS", "1")
hf_login()


## 3. Dataset


In [ ]:
# 3) Build the chat dataset from the package (no logic duplicated here)
import gc, torch
from krona.data import build_dataset
from krona.train import load_base

# A throwaway load just to get the tokenizer for chat-template rendering,
# then release it so both training arms start from a clean card.
_model, tok = load_base(cfg)
del _model
gc.collect(); torch.cuda.empty_cache()

train_ds, eval_ds = build_dataset(cfg, tok)
print(f"train: {len(train_ds)} rows | eval: {len(eval_ds)} rows")
print("\n--- example ---\n", train_ds[0]["text"][:300])


## 4. Train A (LoRA) and B (KronA)

Both arms share data, seed, steps, learning rate, batch size and target modules — only the
adapter type differs. Each starts from a **fresh** base model so they stay independent.

The configs are built separately in `krona/adapters.py` because `LoraConfig` and `LoKrConfig`
do **not** share a shape (`lora_alpha`/`bias` vs `alpha`/`rank_dropout`).


In [ ]:
# 4) Run both adapters — this is the only line that does the work
from krona.train import run_all

results = run_all(cfg, train_ds, eval_ds)


## 5. Compare & save


In [ ]:
# 5) Compare, show samples, persist metrics
from krona.report import summarize

summarize(results, cfg)
